# CityBrain — Full Pipeline
**Vancouver Pavement Risk Assessment | COMP 9130 Group 5**

One notebook: Data → Features → Train → Evaluate → Ablation → XGBoost Baseline

---

In [2]:
# ============================================================
# 0. Setup
# ============================================================
import os, ast, json, warnings, pickle
import numpy as np
import pandas as pd
import geopandas as gpd
from scipy.spatial import cKDTree
from shapely.geometry import shape, Point
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, classification_report
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, Dataset
warnings.filterwarnings('ignore')

# --- CONFIG ---
# For Colab: mount drive first, then set DATA_DIR
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = '/content/drive/MyDrive/AI-FinalProject/data'

DATA_DIR = './data'  # local
SNAP_RADIUS_M = 150
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'mps' if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available() else 'cpu')
print(f'Device: {DEVICE}')

LABEL_MAP = {'VERY GOOD': 0, 'GOOD': 0, 'FAIR': 1, 'POOR': 2, 'VERY POOR': 2}

def safe_load(filename, **kwargs):
    path = os.path.join(DATA_DIR, filename)
    with open(path, 'r', encoding='utf-8', errors='replace') as f:
        first = f.readline()
    sep = ';' if first.count(';') > first.count(',') else ','
    return pd.read_csv(path, sep=sep, on_bad_lines='skip', **kwargs)

Mounted at /content/drive
Device: cpu


## 1. Load & Merge Pavement Data

In [3]:
enriched_path = os.path.join(DATA_DIR, '/content/drive/MyDrive/AI-FinalProject/data/pavement_enriched.csv')
if os.path.exists(enriched_path):
    df = pd.read_csv(enriched_path)
    print(f'Loaded pavement_enriched.csv: {len(df):,} rows')
else:
    df_local = safe_load('/content/drive/MyDrive/AI-FinalProject/data/pavement_condition.csv')
    df_local['road_type'] = 'local'
    df_major = safe_load('/content/drive/MyDrive/AI-FinalProject/data/pavement_condition_major_road_network.csv')
    df_major['road_type'] = 'major'
    df = pd.concat([df_local, df_major], ignore_index=True)
    coords = df['geo_point_2d'].str.split(',', expand=True).astype(float)
    df['lat'], df['lon'] = coords[0], coords[1]
    print(f'Merged from raw: {len(df):,} rows')

df = df[df['PCI Rating'].isin(LABEL_MAP)].copy()
df['risk_label'] = df['PCI Rating'].map(LABEL_MAP)
df['source'] = (df['road_type'] == 'major').astype(int)
print(f'After label mapping: {len(df):,}')
print(df['risk_label'].value_counts().sort_index())

Loaded pavement_enriched.csv: 13,764 rows
After label mapping: 13,764
risk_label
0    6080
1    3061
2    4623
Name: count, dtype: int64


## 2. Feature Engineering — Static Features

In [4]:
pave_coords = np.column_stack([df['lat'].values, df['lon'].values])

# --- streetuse from public_streets.csv ---
df_st = pd.read_csv(os.path.join(DATA_DIR, '/content/drive/MyDrive/AI-FinalProject/data/public_streets.csv'), quotechar='"', on_bad_lines='skip')
st_geo = df_st['geo_point_2d'].dropna().apply(ast.literal_eval)
st_coords = np.array([[d['lat'], d['lon']] for d in st_geo])
_, idx = cKDTree(st_coords).query(pave_coords)
df['streetuse'] = df_st.loc[st_geo.index, 'streetuse'].values[idx]
print('streetuse:', df['streetuse'].value_counts().to_dict())

# --- ROW width ---
df_row = safe_load('/content/drive/MyDrive/AI-FinalProject/data/right_of_way_widths.csv')
geo_col = [c for c in df_row.columns if 'geo_point' in c.lower()][0]
width_col = [c for c in df_row.columns if 'width' in c.lower()][0]
row_geo = df_row[geo_col].dropna().apply(lambda s: ast.literal_eval(s) if '{' in str(s) else s)
row_coords = np.array([[d['lat'], d['lon']] for d in row_geo if isinstance(d, dict)])
row_widths = pd.to_numeric(df_row.loc[row_geo.index[:len(row_coords)], width_col], errors='coerce').fillna(0).values
_, idx = cKDTree(row_coords).query(pave_coords)
df['ROW_width'] = row_widths[idx]
print(f'ROW_width: mean={df["ROW_width"].mean():.1f}')

# --- repair_count ---
df['repair_count'] = 0
repair_path = os.path.join(DATA_DIR, '/content/drive/MyDrive/AI-FinalProject/data/city_project_package_street.csv')
if os.path.exists(repair_path):
    df_repair = safe_load('/content/drive/MyDrive/AI-FinalProject/data/city_project_package_street.csv')
    geo_cols = [c for c in df_repair.columns if 'geo_point' in c.lower()]
    if geo_cols:
        try:
            rg = df_repair[geo_cols[0]].dropna()
            parsed = rg.apply(lambda s: ast.literal_eval(s) if '{' in str(s) else s)
            rc = np.array([[d['lat'], d['lon']] for d in parsed if isinstance(d, dict)])
            LAT_M, LON_M = 111_000, 73_000
            counts = cKDTree(rc * [LAT_M, LON_M]).query_ball_point(pave_coords * [LAT_M, LON_M], r=100)
            df['repair_count'] = [len(c) for c in counts]
        except: pass
print(f'repair_count: nonzero={( df["repair_count"]>0).sum()}')

# --- segment_density (from teammate's approach) ---
tree_all = cKDTree(pave_coords)
neighbours = tree_all.query_ball_point(pave_coords, r=0.003)  # ~300m
df['segment_density'] = [len(n) - 1 for n in neighbours]
print(f'segment_density: mean={df["segment_density"].mean():.1f}')

# --- elevation & slope (from enriched, or zeros) ---
if 'elevation_m' not in df.columns:
    df['elevation_m'] = 0.0
    df['slope_pct'] = 0.0
df['elevation_m'] = df['elevation_m'].fillna(df['elevation_m'].median())
df['slope_pct'] = df['slope_pct'].fillna(df['slope_pct'].median())

# --- neighbourhood ---
df['neighbourhood'] = df.get('neighbourhood', pd.Series('Unknown', index=df.index)).fillna('Unknown')

print(f'\nStatic features done. Shape: {df.shape}')

streetuse: {'Arterial': 5877, 'Residential': 5180, 'Collector': 2707}
ROW_width: mean=41.9
repair_count: nonzero=59
segment_density: mean=35.7

Static features done. Shape: (13764, 18)


## 3. Train/Val/Test Split (before target-encoding to avoid leakage)

In [5]:
y = df['risk_label'].values
indices = np.arange(len(df))

idx_train, idx_temp = train_test_split(indices, test_size=0.30, random_state=42, stratify=y)
idx_val, idx_test = train_test_split(idx_temp, test_size=0.50, random_state=42, stratify=y[idx_temp])

print(f'Train: {len(idx_train):,}  Val: {len(idx_val):,}  Test: {len(idx_test):,}')
print(f'Train labels: {np.bincount(y[idx_train])}')
print(f'Val   labels: {np.bincount(y[idx_val])}')
print(f'Test  labels: {np.bincount(y[idx_test])}')


Train: 9,634  Val: 2,065  Test: 2,065
Train labels: [4256 2142 3236]
Val   labels: [912 459 694]
Test  labels: [912 460 693]


## 4. Target-Encoded Features (train-only, no leakage)

In [6]:
# neighbourhood_high_risk_pct — computed on TRAIN set only
train_df = df.iloc[idx_train]
hood_risk = train_df.groupby('neighbourhood')['risk_label'].apply(
    lambda x: (x == 2).mean()
).to_dict()
global_high_risk = (train_df['risk_label'] == 2).mean()

df['neigh_high_risk_pct'] = df['neighbourhood'].map(hood_risk).fillna(global_high_risk)
print('neigh_high_risk_pct (top 5):')
print(df.iloc[idx_train].groupby('neighbourhood')['neigh_high_risk_pct'].first().sort_values(ascending=False).head())
print(f'\nLeakage check: computed on train ({len(idx_train):,} rows) only, applied to all via map.')

neigh_high_risk_pct (top 5):
neighbourhood
Shaughnessy        0.505405
West Point Grey    0.451327
Arbutus Ridge      0.430493
Sunset             0.408081
Mount Pleasant     0.387755
Name: neigh_high_risk_pct, dtype: float64

Leakage check: computed on train (9,634 rows) only, applied to all via map.


## 5. Feature Engineering — Temporal (BiLSTM)

In [7]:
# --- Weather ---
df_w = safe_load('/content/drive/MyDrive/AI-FinalProject/data/weather_vancouver.csv')
date_col = [c for c in df_w.columns if 'date' in c.lower() or 'time' in c.lower()][0]
df_w['date'] = pd.to_datetime(df_w[date_col], errors='coerce')
df_w = df_w.dropna(subset=['date'])

col_map = {}
for c in df_w.columns:
    cl = c.lower().strip()
    if 'max temp' in cl and 'flag' not in cl: col_map[c] = 'max_temp'
    elif 'min temp' in cl and 'flag' not in cl: col_map[c] = 'min_temp'
    elif 'total precip' in cl and 'flag' not in cl: col_map[c] = 'total_precip'
    elif 'total snow' in cl and 'flag' not in cl: col_map[c] = 'total_snow'
df_w = df_w.rename(columns=col_map)
for c in ['max_temp','min_temp','total_precip','total_snow']:
    if c in df_w.columns: df_w[c] = pd.to_numeric(df_w[c], errors='coerce').fillna(0)

df_w['freeze_thaw'] = ((df_w['max_temp'] > 0) & (df_w['min_temp'] < 0)).astype(int)
df_w['extreme'] = (df_w['total_precip'] > df_w['total_precip'].quantile(0.95)).astype(int)
df_w['year'], df_w['month'] = df_w['date'].dt.year, df_w['date'].dt.month

monthly_wx = df_w.groupby(['year','month']).agg(
    avg_max_temp=('max_temp','mean'), avg_min_temp=('min_temp','mean'),
    total_precip_mm=('total_precip','sum'), total_snow_cm=('total_snow','sum'),
    freeze_thaw_days=('freeze_thaw','sum'), extreme_days=('extreme','sum'),
).reset_index()
print(f'Weather: {len(df_w):,} days, {df_w["year"].min()}-{df_w["year"].max()}')

# --- 311 (only need survey years 2020-2021) ---
print('Loading 311...')
# full.csv = 2022-2026 (useless for us), 2009_2021.csv = 2009-2022 (has our survey years)
df_311 = safe_load('/content/drive/MyDrive/AI-FinalProject/data/311_service_requests_2009_2021.csv', low_memory=False)
print(f'  311_service_requests_2009_2021.csv: {len(df_311):,}')

id_cols = [c for c in df_311.columns if 'case' in c.lower() or 'id' in c.lower()]
if id_cols: df_311 = df_311.drop_duplicates(subset=id_cols[0])
print(f'After dedup: {len(df_311):,}')

# Parse dates
date_cols = [c for c in df_311.columns if 'open' in c.lower() or 'timestamp' in c.lower() or 'created' in c.lower()]
if not date_cols:
    date_cols = [c for c in df_311.columns if 'date' in c.lower()]
df_311['date'] = pd.to_datetime(df_311[date_cols[0]], errors='coerce', utc=True)

# Parse coordinates — handle multiple formats
lat_col = [c for c in df_311.columns if c.lower() in ['latitude','lat']]
lon_col = [c for c in df_311.columns if c.lower() in ['longitude','lon']]
geo_col = [c for c in df_311.columns if 'geo_local' in c.lower() or 'geo_point' in c.lower()]

if lat_col and lon_col:
    df_311['c_lat'] = pd.to_numeric(df_311[lat_col[0]], errors='coerce')
    df_311['c_lon'] = pd.to_numeric(df_311[lon_col[0]], errors='coerce')
elif geo_col:
    raw = df_311[geo_col[0]].dropna()
    if '{' in str(raw.iloc[0]):
        parsed = raw.apply(lambda s: ast.literal_eval(s) if isinstance(s, str) else s)
        df_311.loc[raw.index, 'c_lat'] = parsed.apply(lambda d: d.get('lat') if isinstance(d, dict) else None)
        df_311.loc[raw.index, 'c_lon'] = parsed.apply(lambda d: d.get('lon') if isinstance(d, dict) else None)
    else:
        split = raw.str.split(',', expand=True)
        df_311.loc[raw.index, 'c_lat'] = pd.to_numeric(split[0], errors='coerce')
        df_311.loc[raw.index, 'c_lon'] = pd.to_numeric(split[1], errors='coerce')

df_311 = df_311.dropna(subset=['date','c_lat','c_lon'])
df_311['year'], df_311['month'] = df_311['date'].dt.year, df_311['date'].dt.month

# Filter to survey years only (2020 for local, 2021 for major)
survey_years = set(df['Year'].unique())
df_311 = df_311[df_311['year'].isin(survey_years)]
print(f'311 filtered to survey years {survey_years}: {len(df_311):,}')
print(f'311 with coords: {len(df_311):,}')

# Snap to segments
LAT_M, LON_M = 111_000, 73_000
pave_m = pave_coords * [LAT_M, LON_M]
pave_tree = cKDTree(pave_m)
c311_m = np.column_stack([df_311['c_lat'].values * LAT_M, df_311['c_lon'].values * LON_M])
dists, seg_idx = pave_tree.query(c311_m)
df_311['seg_idx'] = seg_idx
df_311 = df_311[dists <= SNAP_RADIUS_M]
print(f'311 within {SNAP_RADIUS_M}m: {len(df_311):,}')

complaint_monthly = df_311.groupby(['seg_idx','year','month']).size().reset_index(name='cnt')


Weather: 2,557 days, 2019-2025
Loading 311...
  311_service_requests_2009_2021.csv: 2,083,091
After dedup: 2,083,091
311 filtered to survey years {np.int64(2020), np.int64(2021)}: 201,859
311 with coords: 201,859
311 within 150m: 197,819


In [8]:
# --- Build temporal tensor (N, 12, 8) ---
N = len(df)
X_temporal = np.zeros((N, 12, 8), dtype=np.float32)
years = df['Year'].values

# Neighbourhood-month totals for density
seg_to_neigh = df['neighbourhood'].values
neigh_month_tot = {}
for _, r in complaint_monthly.iterrows():
    key = (seg_to_neigh[int(r['seg_idx'])], int(r['year']), int(r['month']))
    neigh_month_tot[key] = neigh_month_tot.get(key, 0) + r['cnt']

for i in range(N):
    yr = years[i]
    wx = monthly_wx[monthly_wx['year'] == yr]
    for _, r in wx.iterrows():
        m = int(r['month']) - 1
        if 0 <= m < 12:
            X_temporal[i,m,0] = r['avg_max_temp']
            X_temporal[i,m,1] = r['avg_min_temp']
            X_temporal[i,m,2] = r['total_precip_mm']
            X_temporal[i,m,3] = r['total_snow_cm']
            X_temporal[i,m,4] = r['freeze_thaw_days']
            X_temporal[i,m,7] = r['extreme_days']

    seg_c = complaint_monthly[(complaint_monthly['seg_idx']==i) & (complaint_monthly['year']==yr)]
    neigh = seg_to_neigh[i]
    for _, r in seg_c.iterrows():
        m = int(r['month']) - 1
        if 0 <= m < 12:
            X_temporal[i,m,5] = r['cnt']
            nt = neigh_month_tot.get((neigh, yr, int(r['month'])), 1)
            X_temporal[i,m,6] = min(r['cnt'] / max(nt, 1), 1.0)

# Also store total complaints as static feature
df['complaint_total'] = X_temporal[:,:,5].sum(axis=1)

print(f'X_temporal: {X_temporal.shape}')
print(f'Nonzero complaint-months: {(X_temporal[:,:,5]>0).sum():,}')
print(f'complaint_total: mean={df["complaint_total"].mean():.1f}, max={df["complaint_total"].max():.0f}')

X_temporal: (13764, 12, 8)
Nonzero complaint-months: 50,262
complaint_total: mean=7.3, max=626


## 6. Assemble Branch Tensors & Normalize

In [9]:
# --- Road-MLP: 9 dims ---
su_dummies = pd.get_dummies(df['streetuse'], prefix='su')
for c in ['su_Arterial','su_Collector','su_Residential']:
    if c not in su_dummies.columns: su_dummies[c] = 0

X_road_raw = np.column_stack([
    df['length_(m)'].fillna(df['length_(m)'].median()).values,
    su_dummies[['su_Arterial','su_Collector','su_Residential']].values,
    df['elevation_m'].values,
    df['slope_pct'].values,
    df['repair_count'].values,
    df['segment_density'].values,
    df['source'].values,
]).astype(np.float32)

# --- Tabular-MLP: ~27 dims ---
neigh_dummies = pd.get_dummies(df['neighbourhood'], prefix='n')
X_tab_raw = np.column_stack([
    neigh_dummies.values,
    df['ROW_width'].values,
    df['complaint_total'].values,
    df['neigh_high_risk_pct'].values,
]).astype(np.float32)

# --- Normalize (fit on train only) ---
sc_road = StandardScaler().fit(X_road_raw[idx_train])
sc_tab = StandardScaler().fit(X_tab_raw[idx_train])
sc_temp = StandardScaler().fit(X_temporal[idx_train].reshape(-1, 8))

def split_norm(X, sc, idxs, seq=False):
    if seq:
        n = len(idxs)
        return sc.transform(X[idxs].reshape(-1,8)).reshape(n,12,8).astype(np.float32)
    return sc.transform(X[idxs]).astype(np.float32)

Xr_tr, Xr_v, Xr_te = split_norm(X_road_raw,sc_road,idx_train), split_norm(X_road_raw,sc_road,idx_val), split_norm(X_road_raw,sc_road,idx_test)
Xt_tr, Xt_v, Xt_te = split_norm(X_temporal,sc_temp,idx_train,True), split_norm(X_temporal,sc_temp,idx_val,True), split_norm(X_temporal,sc_temp,idx_test,True)
Xb_tr, Xb_v, Xb_te = split_norm(X_tab_raw,sc_tab,idx_train), split_norm(X_tab_raw,sc_tab,idx_val), split_norm(X_tab_raw,sc_tab,idx_test)
y_tr, y_v, y_te = y[idx_train], y[idx_val], y[idx_test]

print(f'Road:     {Xr_tr.shape}')
print(f'Temporal: {Xt_tr.shape}')
print(f'Tabular:  {Xb_tr.shape}')
print(f'Labels:   {y_tr.shape}')

Road:     (9634, 9)
Temporal: (9634, 12, 8)
Tabular:  (9634, 26)
Labels:   (9634,)


## 7. Model Definitions

In [10]:
class RoadMLP(nn.Module):
    def __init__(self, dim, emb=16, drop=0.3):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(dim,64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(drop),
                                 nn.Linear(64,32), nn.ReLU(), nn.Linear(32,emb), nn.ReLU())
        self.head = nn.Linear(emb, 3)
    def get_embedding(self, x): return self.enc(x)
    def forward(self, x): return self.head(self.enc(x))

class BiLSTMBranch(nn.Module):
    def __init__(self, feat=8, hidden=64, layers=2, drop=0.3):
        super().__init__()
        self.lstm = nn.LSTM(feat, hidden, layers, batch_first=True, bidirectional=True,
                            dropout=drop if layers > 1 else 0)
        self.fc = nn.Sequential(nn.Linear(hidden*2, 128), nn.ReLU(), nn.Dropout(drop))
        self.head = nn.Linear(128, 3)
    def get_embedding(self, x):
        _, (h, _) = self.lstm(x)
        return self.fc(torch.cat([h[-2], h[-1]], dim=1))
    def forward(self, x): return self.head(self.get_embedding(x))

class TabularMLP(nn.Module):
    def __init__(self, dim, emb=16, drop=0.3):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(dim,64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(drop),
                                 nn.Linear(64,32), nn.ReLU(), nn.Linear(32,emb), nn.ReLU())
        self.head = nn.Linear(emb, 3)
    def get_embedding(self, x): return self.enc(x)
    def forward(self, x): return self.head(self.enc(x))

class CityBrainFusion(nn.Module):
    def __init__(self, road, bilstm, tabular, drop=0.3):
        super().__init__()
        self.road, self.bilstm, self.tabular = road, bilstm, tabular
        self.fusion = nn.Sequential(
            nn.Linear(160, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(drop),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(drop),
            nn.Linear(64, 3))
    def forward(self, xr, xt, xb):
        return self.fusion(torch.cat([
            self.road.get_embedding(xr),
            self.bilstm.get_embedding(xt),
            self.tabular.get_embedding(xb)], dim=1))
    def forward_ablation(self, xr, xt, xb, disable=None):
        er = self.road.get_embedding(xr)    if disable != 'road'    else torch.zeros_like(self.road.get_embedding(xr))
        et = self.bilstm.get_embedding(xt)  if disable != 'bilstm'  else torch.zeros_like(self.bilstm.get_embedding(xt))
        eb = self.tabular.get_embedding(xb) if disable != 'tabular' else torch.zeros_like(self.tabular.get_embedding(xb))
        return self.fusion(torch.cat([er, et, eb], dim=1))

print('Models defined: RoadMLP, BiLSTMBranch, TabularMLP, CityBrainFusion')

Models defined: RoadMLP, BiLSTMBranch, TabularMLP, CityBrainFusion


## 8. Training Utilities

In [11]:
class FusionDS(Dataset):
    def __init__(self, xr, xt, xb, y):
        self.xr, self.xt, self.xb, self.y = xr, xt, xb, y
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.xr[i], self.xt[i], self.xb[i], self.y[i]

# Class weights
cc = np.bincount(y_tr)
CW = torch.FloatTensor((1.0/cc) / (1.0/cc).sum() * 3).to(DEVICE)
print(f'Class weights: {CW.cpu().numpy()}')

def train_branch(model, X_tr, X_v, y_tr_, y_v_, epochs=50, bs=256, lr=1e-3, patience=10):
    """Train a single branch model."""
    model = model.to(DEVICE)
    tr_dl = DataLoader(TensorDataset(torch.tensor(X_tr), torch.tensor(y_tr_).long()), batch_size=bs, shuffle=True)
    v_dl  = DataLoader(TensorDataset(torch.tensor(X_v),  torch.tensor(y_v_).long()),  batch_size=bs)
    crit = nn.CrossEntropyLoss(weight=CW)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    best_f1, wait, best_sd = 0, 0, None
    for ep in range(1, epochs+1):
        model.train()
        for X, yy in tr_dl:
            X, yy = X.to(DEVICE), yy.to(DEVICE)
            opt.zero_grad(); loss = crit(model(X), yy); loss.backward(); opt.step()
        sch.step()
        model.eval()
        preds, labs = [], []
        with torch.no_grad():
            for X, yy in v_dl:
                preds.extend(model(X.to(DEVICE)).argmax(1).cpu().numpy())
                labs.extend(yy.numpy())
        f1 = f1_score(labs, preds, average='macro')
        if f1 > best_f1: best_f1, wait, best_sd = f1, 0, {k:v.cpu().clone() for k,v in model.state_dict().items()}
        else: wait += 1
        if ep % 10 == 0: print(f'  Ep {ep:>3} val_f1={f1:.4f} best={best_f1:.4f}')
        if wait >= patience: print(f'  Early stop ep {ep}'); break
    model.load_state_dict(best_sd)
    return model, best_f1

def eval_test(model, X_te, y_te_, bs=256):
    """Evaluate a branch on test set."""
    model.eval()
    dl = DataLoader(TensorDataset(torch.tensor(X_te), torch.tensor(y_te_).long()), batch_size=bs)
    preds, labs = [], []
    with torch.no_grad():
        for X, yy in dl:
            preds.extend(model(X.to(DEVICE)).argmax(1).cpu().numpy())
            labs.extend(yy.numpy())
    return f1_score(labs, preds, average='macro'), np.array(preds), np.array(labs)

print('Training utilities ready.')

Class weights: [0.6973287 1.3855419 0.9171294]
Training utilities ready.


## 9. Pre-train Individual Branches

In [12]:
print('=== Road-MLP ===')
road_model = RoadMLP(dim=Xr_tr.shape[1])
road_model, road_val_f1 = train_branch(road_model, Xr_tr, Xr_v, y_tr, y_v)
road_test_f1, _, _ = eval_test(road_model, Xr_te, y_te)
print(f'  Road-MLP: val={road_val_f1:.4f} test={road_test_f1:.4f}\n')

print('=== BiLSTM ===')
bilstm_model = BiLSTMBranch(feat=8)
bilstm_model, bilstm_val_f1 = train_branch(bilstm_model, Xt_tr, Xt_v, y_tr, y_v)
bilstm_test_f1, _, _ = eval_test(bilstm_model, Xt_te, y_te)
print(f'  BiLSTM: val={bilstm_val_f1:.4f} test={bilstm_test_f1:.4f}\n')

print('=== Tabular-MLP ===')
tab_model = TabularMLP(dim=Xb_tr.shape[1])
tab_model, tab_val_f1 = train_branch(tab_model, Xb_tr, Xb_v, y_tr, y_v)
tab_test_f1, _, _ = eval_test(tab_model, Xb_te, y_te)
print(f'  Tabular-MLP: val={tab_val_f1:.4f} test={tab_test_f1:.4f}\n')

print('\n--- Unimodal Summary ---')
print(f'{"Branch":<15} {"Val F1":>8} {"Test F1":>8}')
print('-'*33)
for name, vf, tf in [('Road-MLP',road_val_f1,road_test_f1),('BiLSTM',bilstm_val_f1,bilstm_test_f1),('Tabular-MLP',tab_val_f1,tab_test_f1)]:
    print(f'{name:<15} {vf:>8.4f} {tf:>8.4f}')

=== Road-MLP ===
  Ep  10 val_f1=0.3865 best=0.3945
  Ep  20 val_f1=0.3978 best=0.3987
  Early stop ep 23
  Road-MLP: val=0.3987 test=0.3758

=== BiLSTM ===
  Ep  10 val_f1=0.3553 best=0.3553
  Ep  20 val_f1=0.3395 best=0.3553
  Early stop ep 20
  BiLSTM: val=0.3553 test=0.3230

=== Tabular-MLP ===
  Ep  10 val_f1=0.3841 best=0.3873
  Early stop ep 17
  Tabular-MLP: val=0.3873 test=0.3815


--- Unimodal Summary ---
Branch            Val F1  Test F1
---------------------------------
Road-MLP          0.3987   0.3758
BiLSTM            0.3553   0.3230
Tabular-MLP       0.3873   0.3815


## 10. Fusion Training

In [13]:
fusion = CityBrainFusion(road_model, bilstm_model, tab_model).to(DEVICE)
print(f'Fusion params: {sum(p.numel() for p in fusion.parameters()):,}')

tr_dl = DataLoader(FusionDS(torch.tensor(Xr_tr),torch.tensor(Xt_tr),torch.tensor(Xb_tr),torch.tensor(y_tr).long()), batch_size=256, shuffle=True)
v_dl  = DataLoader(FusionDS(torch.tensor(Xr_v),torch.tensor(Xt_v),torch.tensor(Xb_v),torch.tensor(y_v).long()), batch_size=256)
te_dl = DataLoader(FusionDS(torch.tensor(Xr_te),torch.tensor(Xt_te),torch.tensor(Xb_te),torch.tensor(y_te).long()), batch_size=256)

crit = nn.CrossEntropyLoss(weight=CW)

def eval_fusion(model, dl, disable=None):
    model.eval()
    preds, labs = [], []
    with torch.no_grad():
        for xr,xt,xb,yy in dl:
            xr,xt,xb,yy = xr.to(DEVICE),xt.to(DEVICE),xb.to(DEVICE),yy.to(DEVICE)
            logits = model.forward_ablation(xr,xt,xb,disable) if disable else model(xr,xt,xb)
            preds.extend(logits.argmax(1).cpu().numpy()); labs.extend(yy.cpu().numpy())
    return f1_score(labs, preds, average='macro'), np.array(preds), np.array(labs)

# Phase 1: Warmup (branches frozen)
print('\n--- Phase 1: Warmup (5 ep, branches frozen) ---')
for p in list(fusion.road.parameters()) + list(fusion.bilstm.parameters()) + list(fusion.tabular.parameters()):
    p.requires_grad = False
opt = torch.optim.AdamW(fusion.fusion.parameters(), lr=1e-3, weight_decay=1e-4)
for ep in range(1, 6):
    fusion.train()
    for xr,xt,xb,yy in tr_dl:
        xr,xt,xb,yy = xr.to(DEVICE),xt.to(DEVICE),xb.to(DEVICE),yy.to(DEVICE)
        opt.zero_grad(); crit(fusion(xr,xt,xb), yy).backward(); opt.step()
    f1, _, _ = eval_fusion(fusion, v_dl)
    print(f'  Warmup {ep}/5 val_f1={f1:.4f}')

# Phase 2: End-to-end
print('\n--- Phase 2: End-to-end (50 ep) ---')
for p in fusion.parameters(): p.requires_grad = True
opt = torch.optim.AdamW(fusion.parameters(), lr=1e-4, weight_decay=1e-4)
sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=50)
best_f1, wait, best_sd = 0, 0, None
for ep in range(1, 51):
    fusion.train()
    for xr,xt,xb,yy in tr_dl:
        xr,xt,xb,yy = xr.to(DEVICE),xt.to(DEVICE),xb.to(DEVICE),yy.to(DEVICE)
        opt.zero_grad(); loss = crit(fusion(xr,xt,xb), yy); loss.backward()
        nn.utils.clip_grad_norm_(fusion.parameters(), 1.0); opt.step()
    sch.step()
    f1, _, _ = eval_fusion(fusion, v_dl)
    mk = ''
    if f1 > best_f1: best_f1, wait, best_sd = f1, 0, {k:v.cpu().clone() for k,v in fusion.state_dict().items()}; mk = ' *'
    else: wait += 1
    if ep % 5 == 0: print(f'  Ep {ep:>3} val_f1={f1:.4f} best={best_f1:.4f}{mk}')
    if wait >= 10: print(f'  Early stop ep {ep}'); break

fusion.load_state_dict(best_sd)
fusion = fusion.to(DEVICE)

Fusion params: 191,372

--- Phase 1: Warmup (5 ep, branches frozen) ---
  Warmup 1/5 val_f1=0.3442
  Warmup 2/5 val_f1=0.3822
  Warmup 3/5 val_f1=0.4003
  Warmup 4/5 val_f1=0.4012
  Warmup 5/5 val_f1=0.3915

--- Phase 2: End-to-end (50 ep) ---
  Ep   5 val_f1=0.3991 best=0.3991 *
  Ep  10 val_f1=0.4063 best=0.4063 *
  Ep  15 val_f1=0.4069 best=0.4070
  Ep  20 val_f1=0.4138 best=0.4150
  Ep  25 val_f1=0.4133 best=0.4170
  Ep  30 val_f1=0.4150 best=0.4170
  Ep  35 val_f1=0.4163 best=0.4184
  Ep  40 val_f1=0.4163 best=0.4184
  Early stop ep 41


## 11. Test Evaluation & Ablation

In [14]:
test_f1, preds, labs = eval_fusion(fusion, te_dl)
print(f'\n{"="*55}')
print(f'FUSION TEST MACRO F1: {test_f1:.4f}  (val best: {best_f1:.4f})')
print(f'{"="*55}')
print(classification_report(labs, preds, target_names=['Low','Medium','High']))

# Ablation
print(f'{"="*55}')
print('ABLATION STUDY')
print(f'{"="*55}')
results = {'Full model': test_f1}
for branch in ['road','bilstm','tabular']:
    ab_f1, _, _ = eval_fusion(fusion, te_dl, disable=branch)
    results[f'w/o {branch}'] = ab_f1

print(f'\n{"Config":<20} {"F1":>6} {"Delta":>8}')
print('-'*36)
for k, v in results.items():
    d = f'{v - test_f1:+.4f}' if k != 'Full model' else '  base'
    print(f'{k:<20} {v:>6.4f} {d:>8}')

# Summary vs unimodal
print(f'\n--- Fusion vs Unimodal ---')
best_uni = max(road_test_f1, bilstm_test_f1, tab_test_f1)
print(f'Best unimodal: {best_uni:.4f}')
print(f'Fusion:        {test_f1:.4f}')
print(f'Improvement:   {test_f1 - best_uni:+.4f}')



FUSION TEST MACRO F1: 0.3915  (val best: 0.4184)
              precision    recall  f1-score   support

         Low       0.54      0.40      0.46       912
      Medium       0.26      0.40      0.32       460
        High       0.40      0.40      0.40       693

    accuracy                           0.40      2065
   macro avg       0.40      0.40      0.39      2065
weighted avg       0.43      0.40      0.41      2065

ABLATION STUDY

Config                   F1    Delta
------------------------------------
Full model           0.3915     base
w/o road             0.3447  -0.0469
w/o bilstm           0.3933  +0.0017
w/o tabular          0.3132  -0.0783

--- Fusion vs Unimodal ---
Best unimodal: 0.3815
Fusion:        0.3915
Improvement:   +0.0101


## 12. XGBoost Baseline

In [15]:
try:
    from xgboost import XGBClassifier
except ImportError:
    !pip install -q xgboost
    from xgboost import XGBClassifier

# Combine all features into flat vector
X_flat_tr = np.concatenate([Xr_tr, Xt_tr.reshape(len(Xr_tr),-1), Xb_tr], axis=1)
X_flat_v  = np.concatenate([Xr_v,  Xt_v.reshape(len(Xr_v),-1),  Xb_v],  axis=1)
X_flat_te = np.concatenate([Xr_te, Xt_te.reshape(len(Xr_te),-1), Xb_te], axis=1)

xgb = XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric='mlogloss', random_state=42, verbosity=0
)
xgb.fit(X_flat_tr, y_tr, eval_set=[(X_flat_v, y_v)], verbose=False)

xgb_preds = xgb.predict(X_flat_te)
xgb_f1 = f1_score(y_te, xgb_preds, average='macro')

print(f'\nXGBoost Test Macro F1: {xgb_f1:.4f}')
print(classification_report(y_te, xgb_preds, target_names=['Low','Medium','High']))

print(f'\n--- Final Comparison ---')
print(f'{"Model":<20} {"Test F1":>8}')
print('-'*30)
print(f'{"Road-MLP":<20} {road_test_f1:>8.4f}')
print(f'{"BiLSTM":<20} {bilstm_test_f1:>8.4f}')
print(f'{"Tabular-MLP":<20} {tab_test_f1:>8.4f}')
print(f'{"Fusion (ours)":<20} {test_f1:>8.4f}')
print(f'{"XGBoost baseline":<20} {xgb_f1:>8.4f}')


XGBoost Test Macro F1: 0.4010
              precision    recall  f1-score   support

         Low       0.52      0.70      0.60       912
      Medium       0.29      0.11      0.16       460
        High       0.46      0.44      0.45       693

    accuracy                           0.48      2065
   macro avg       0.42      0.42      0.40      2065
weighted avg       0.45      0.48      0.45      2065


--- Final Comparison ---
Model                 Test F1
------------------------------
Road-MLP               0.3758
BiLSTM                 0.3230
Tabular-MLP            0.3815
Fusion (ours)          0.3915
XGBoost baseline       0.4010


---
**End of pipeline.** All results above are reproducible (random_state=42).